In [28]:
#Order Level Dataset
import pandas as pd
import numpy as np

DATA_PATH = "C:\\FINAL YEAR CS PROJECTS\\End-to-End-ECommerce-Intelligence-System\\data\\raw\\"

customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        dayfirst=True,
        errors="coerce"
    )

orders[date_columns].dtypes

orders = orders.rename(columns={
    "order_purchase_timestamp": "purchase_timestamp",
    "order_approved_at": "approved_at",
    "order_delivered_carrier_date": "delivered_carrier_date",
    "order_delivered_customer_date": "delivered_customer_date",
    "order_estimated_delivery_date": "estimated_delivery_date"
})

orders_delivered = orders[orders["order_status"] == "delivered"].copy()

orders_delivered.shape

orders_delivered["order_status"].value_counts()

orders_delivered.isnull().sum()

orders_delivered = orders_delivered.dropna(subset=[
    "purchase_timestamp",
    "delivered_customer_date",
    "estimated_delivery_date"
])

orders_delivered.isnull().sum()

#Order items aggregation
order_items_agg = order_items.groupby("order_id").agg(
    product_count=("product_id", "count"),
    unique_product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_price=("price", "mean")
).reset_index()

order_items_agg.head()

order_items_agg["order_id"].is_unique


#Add Main Product Category Per Order
products_translated = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

order_items_products = order_items.merge(
    products_translated[["product_id", "product_category_name_english"]],
    on="product_id",
    how="left"
)

order_category = (
    order_items_products
    .groupby("order_id")["product_category_name_english"]
    .agg(lambda x: x.mode()[0] if not x.mode().empty else "unknown")
    .reset_index()
)

order_category = order_category.rename(columns={
    "product_category_name_english": "main_product_category"
})

order_category.head()


#Order_paymenta aggregation
order_payments_agg = order_payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", lambda x: x.mode()[0] if not x.mode().empty else "unknown")
).reset_index()

order_payments_agg.head()


#Order_reviews aggregation
order_reviews_agg = order_reviews.groupby("order_id").agg(
    review_score=("review_score", "mean")
).reset_index()

order_reviews_agg.head()


#Merge orders with customers
order_level = orders_delivered.merge(
    customers,
    on="customer_id",
    how="left"
)

order_level.shape

order_level["order_id"].is_unique


#Merge aggregated tables
order_level = order_level.merge(
    order_items_agg,
    on="order_id",
    how="left"
)

order_level = order_level.merge(
    order_category,
    on="order_id",
    how="left"
)

order_level = order_level.merge(
    order_payments_agg,
    on="order_id",
    how="left"
)

order_level = order_level.merge(
    order_reviews_agg,
    on="order_id",
    how="left"
)

order_level.head()

order_level["order_id"].is_unique

order_level.shape


#Delivery features

#Delivery days
order_level["delivery_days"] = (
    order_level["delivered_customer_date"] - order_level["purchase_timestamp"]
).dt.days

#Estimated delivery days
order_level["estimated_delivery_days"] = (
    order_level["estimated_delivery_date"] - order_level["purchase_timestamp"]
).dt.days

#Delay days
order_level["delay_days"] = (
    order_level["delivered_customer_date"] - order_level["estimated_delivery_date"]
).dt.days

#Late targets
order_level["is_late"] = np.where(order_level["delay_days"] > 0, 1, 0)

order_level[["delivery_days", "estimated_delivery_days", "delay_days", "is_late"]].head()


#Time features
order_level["purchase_year"] = order_level["purchase_timestamp"].dt.year
order_level["purchase_month"] = order_level["purchase_timestamp"].dt.month
order_level["purchase_day"] = order_level["purchase_timestamp"].dt.day
order_level["purchase_day_of_week"] = order_level["purchase_timestamp"].dt.dayofweek
order_level["purchase_hour"] = order_level["purchase_timestamp"].dt.hour
order_level["purchase_year_month"] = order_level["purchase_timestamp"].dt.to_period("M").astype(str)

order_level[["purchase_year", "purchase_month", "purchase_day", "purchase_day_of_week", "purchase_hour", "purchase_year_month"]].head()


#Handling missing values
#Hadling review scores
order_level["review_score"] = order_level["review_score"].fillna(
    order_level["review_score"].median()
)

#Handling payment types
order_level["payment_type"] = order_level["payment_type"].fillna("unknown")

#Handling total payment
order_level["total_payment"] = order_level["total_payment"].fillna(0)

#Handling payment installments
order_level["payment_installments"] = order_level["payment_installments"].fillna(0)

#Handling approved at
order_level["approved_at"] = order_level["approved_at"].fillna(
    order_level["purchase_timestamp"]
)

#Handling delivered carrier dates
order_level["approved_at"] = order_level["approved_at"].fillna(
    order_level["purchase_timestamp"]
)

#Filling numeric values
numeric_cols = [
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments"
]

for col in numeric_cols:
    order_level[col] = order_level[col].fillna(0)

#Final columns
final_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "order_status",
    "purchase_timestamp",
    "approved_at",
    "delivered_carrier_date",
    "delivered_customer_date",
    "estimated_delivery_date",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "is_late",
    "product_count",
    "unique_product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "avg_price",
    "total_payment",
    "payment_installments",
    "payment_type",
    "review_score",
    "main_product_category",
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_day_of_week",
    "purchase_hour",
    "purchase_year_month"
]

order_level = order_level[final_columns]

order_level["order_id"].is_unique

order_level.shape

order_level["is_late"].value_counts(normalize=True) * 100

order_level["review_score"].describe()

order_level["review_score"].value_counts().sort_index()

order_level["bad_review"] = np.where(
    order_level["review_score"] <= 2,
    1,
    0
)

order_level["bad_review"].value_counts(normalize=True) * 100

order_level["total_payment"].sum()

order_level.to_csv("../data/processed/order_level_dataset.csv", index=False)

pd.read_csv("../data/processed/order_level_dataset.csv").head()

phase_2_summary = {
    "final_rows": order_level.shape[0],
    "final_columns": order_level.shape[1],
    "unique_orders": order_level["order_id"].nunique(),
    "total_revenue": order_level["total_payment"].sum(),
    "late_delivery_rate": order_level["is_late"].mean(),
    "avg_review_score": order_level["review_score"].mean(),
    "avg_delivery_days": order_level["delivery_days"].mean()
}

phase_2_summary


pd.DataFrame([phase_2_summary]).to_csv(
    "../reports/phase_2_order_level_summary.csv",
    index=False
)


# Phase 2 Conclusion

In Phase 2, the raw Olist datasets were cleaned and transformed into an order-level analytical dataset.

Main work completed:

- Converted order timestamp columns into datetime format.
- Filtered delivered orders for reliable delivery analysis.
- Aggregated order items to maintain one row per order.
- Aggregated payment records to maintain one row per order.
- Aggregated review records to maintain one row per order.
- Added translated product categories.
- Merged customer, order, item, payment, review, and product category information.
- Created delivery features such as delivery_days, estimated_delivery_days, delay_days, and is_late.
- Created purchase time features for trend analysis.
- Saved the final order-level dataset.

The final dataset follows the rule:

ONE ROW = ONE ORDER

This dataset will be used for EDA, SQL warehousing, Power BI dashboarding, and machine learning.